In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

## Problem 1

### (a)

Hold a portfolio long one $(K,T_2)$-forward and short one $(F_t,T_2)$-forward. Its time-$t$ value is
$$
f_t - 0 = f_t,
$$
because the second leg, by definition of forward price, has time-$t$ value 0. Its time-$T_2$ payoff is deterministic at time $t$:
$$
(S_{T_2}-K)-(S_{T_2}-F_t) = F_t-K.
$$
Discount that deterministic payoff to time $t$:
$$
f_t = e^{-r(T_2-t)}\,(F_t-K).
$$

### (b)

The arbitrage requires that we can buy the underlying at time $t$ and **costlessly carry** it to $T_2$ to make delivery. Crude oil is bulky and perishable, so storage incurs real costs (and is constrained by physical capacity); the carrying leg of the strategy is not free, so the simple "buy the stock and short the forward" arbitrage breaks down when "stock" is replaced by "crude oil."

In [2]:
# Exponential Ornstein-Uhlenbeck process

class XOU:

    def __init__(self, kappa, alpha, sigma, S0, r):

        self.kappa = kappa
        self.alpha = alpha
        self.sigma = sigma
        self.S0 = S0
        self.r = r

In [3]:
hw5dynamics=XOU(kappa = 0.472, alpha = 4.4, sigma = 0.368, S0 = 106.9, r = 0.05)

In [4]:
class CallOnForwardPrice:

    def __init__(self, K1, T1, T2):

        self.K1 = K1
        self.T1 = T1
        self.T2 = T2


In [5]:
hw5contract=CallOnForwardPrice(K1 = 103.2, T1 = 0.5, T2 = 0.75)

In [6]:
class MCengine:

    def __init__(self, N, M, epsilon, seed):

        self.N = N   # Number of timesteps on each path
        self.M = M   # Number of paths
        self.epsilon = epsilon  # For the dC/dS calculation
        self.rng = np.random.default_rng(seed=seed) # Seeding the random number generator with a specified number helps make the calculations reproducible

    def price_call_XOU(self, contract, dynamics):

        K, T1, T2 = contract.K1, contract.T1, contract.T2
        kappa, alpha, sigma = dynamics.kappa, dynamics.alpha, dynamics.sigma
        r, S0 = dynamics.r, dynamics.S0

        dt = T1 / self.N
        Z = self.rng.normal(size=(self.M, self.N))

        tau = T2 - T1
        a = np.exp(-kappa*tau)
        b = (1 - a)*alpha + sigma**2/(4*kappa)*(1 - np.exp(-2*kappa*tau))

        def discounted_payoff(S_init):
            X = np.full(self.M, np.log(S_init))
            for n in range(self.N):
                X = X + kappa*(alpha - X)*dt + sigma*np.sqrt(dt)*Z[:, n]
            F_T1 = np.exp(a*X + b)
            return np.exp(-r*T1) * np.maximum(F_T1 - K, 0.0)

        Y0 = discounted_payoff(S0)
        Yeps = discounted_payoff(S0 + self.epsilon)

        call_price = Y0.mean()
        standard_error = Y0.std(ddof=1) / np.sqrt(self.M)
        call_delta = (Yeps.mean() - Y0.mean()) / self.epsilon

        return (call_price, standard_error, call_delta)


In [7]:
hw5MC = MCengine(N=100, M=200000, epsilon=0.01, seed=0)
# Change M if necessary

In [8]:
(call_price, standard_error, call_delta) = hw5MC.price_call_XOU(hw5contract,hw5dynamics)

In [9]:
print(call_price, standard_error, call_delta)

7.788026496905533 0.02983412324816588 0.3415465981566612


### (c) and (d)

Simulate $X = \log S$ by Euler with $\Delta t = T_1/N$:
$$
X^{(m)}_{t_{n+1}} = X^{(m)}_{t_n} + \kappa(\alpha - X^{(m)}_{t_n})\,\Delta t + \sigma\sqrt{\Delta t}\,Z^{(m)}_n,
$$
then
$$
F_{T_1}^{(m)} = \exp\!\Big(e^{-\kappa(T_2-T_1)}X^{(m)}_{T_1} + (1-e^{-\kappa(T_2-T_1)})\alpha + \tfrac{\sigma^2}{4\kappa}(1-e^{-2\kappa(T_2-T_1)})\Big),
$$
and the discounted payoff is $Y^{(m)} = e^{-rT_1}(F^{(m)}_{T_1}-K)^+$. The Monte Carlo estimator and its standard error are
$$
\hat C_M = \tfrac1M\sum Y^{(m)},\qquad
\widehat{\text{SE}} = \frac{\hat\sigma_M}{\sqrt M}.
$$

For (d), $\partial C/\partial S$ is the forward difference $(\hat C_M(S_0+\varepsilon)-\hat C_M(S_0))/\varepsilon$, with the **same** $Z$ matrix reused for both estimators.

With $M=200{,}000$ paths, the standard error is well below the required $0.05$.

### (e)

From part (a), $f_0 = e^{-rT_2}(F_0-K)$, where
$$
F_0 = \exp\!\Big(e^{-\kappa T_2}\log S_0 + (1-e^{-\kappa T_2})\alpha + \tfrac{\sigma^2}{4\kappa}(1-e^{-2\kappa T_2})\Big).
$$
Only the first term inside the exponent depends on $S_0$, with derivative $e^{-\kappa T_2}/S_0$. So
$$
\frac{\partial F_0}{\partial S} = F_0\,\frac{e^{-\kappa T_2}}{S_0},
\qquad
\boxed{\;\frac{\partial f_0}{\partial S} = \frac{e^{-rT_2}\,e^{-\kappa T_2}\,F_0}{S_0}.\;}
$$

### (f)

A hedge portfolio long $n$ forwards has price $n f_t$. To replicate a long call position by continuously rebalancing $n$, match deltas:
$$
\frac{\partial C}{\partial S} = n\,\frac{\partial f_0}{\partial S}
\quad\Longrightarrow\quad
n = \frac{\partial C/\partial S}{\partial f_0/\partial S}.
$$

In [10]:
kappa, alpha, sigma = hw5dynamics.kappa, hw5dynamics.alpha, hw5dynamics.sigma
r, S0 = hw5dynamics.r, hw5dynamics.S0
T1, T2, K = hw5contract.T1, hw5contract.T2, hw5contract.K1

F0 = np.exp(np.exp(-kappa*T2)*np.log(S0) + (1-np.exp(-kappa*T2))*alpha
            + sigma**2/(4*kappa)*(1-np.exp(-2*kappa*T2)))
df0_dS = np.exp(-r*T2)*np.exp(-kappa*T2)*F0/S0
n_hedge = call_delta / df0_dS

print("F0       :", F0)
print("df0/dS   :", df0_dS)
print("n_hedge  :", n_hedge)

F0       : 102.23035226277118
df0/dS   : 0.6465109164314244
n_hedge  : 0.5282920821227759


### (g)

The contract pays $\theta(S_{T_2}-K)$ at $T_2$, with $\theta\in\{4000,5000\}$ chosen at $T_1$. By iterated expectations under the risk-neutral measure,
$$
V_0 = e^{-rT_2}\,\mathbb E\big[\theta^*\,\mathbb E_{T_1}(S_{T_2}-K)\big]
    = e^{-rT_2}\,\mathbb E\big[\theta^*(F_{T_1}-K)\big].
$$
Holder optimality: $\theta^*=5000$ if $F_{T_1}>K$ and $\theta^*=4000$ if $F_{T_1}\le K$. So
$$
\theta^*(F_{T_1}-K) = 4000(F_{T_1}-K) + 1000(F_{T_1}-K)^+.
$$
Using $\mathbb E[F_{T_1}]=F_0$ (forward price is a $\mathbb Q$-martingale) and $C = e^{-rT_1}\mathbb E[(F_{T_1}-K)^+]$,
$$
\boxed{\;V_0 = 4000\,e^{-rT_2}(F_0-K) + 1000\,e^{-r(T_2-T_1)}\,C.\;}
$$

In [11]:
V0 = 4000*np.exp(-r*T2)*(F0-K) + 1000*np.exp(-r*(T2-T1))*call_price
print("Purchase agreement value V0:", V0)

Purchase agreement value V0: 3955.44492738431


## Problem 2

### (a)

By L2.29, since $\sigma$ depends only on $t$,
$$
\log S_T \sim \text{Normal}\!\left(\log S_0 + \big(r - \tfrac12\bar\sigma_T^2\big)T,\;\bar\sigma_T^2\,T\right),
\qquad
\bar\sigma_T := \sqrt{\frac{1}{T}\int_0^T \sigma^2(t)\,dt}.
$$
So time-0 European call prices satisfy $C(K,T) = C_{BS}(\bar\sigma_T)$, hence $\sigma_{\text{imp}}(K,T) = \bar\sigma_T$.

- **Term structure (vs $T$):** Yes. $\bar\sigma_T$ varies with $T$ whenever $\sigma(t)$ is non-constant.
- **Skew (vs $K$):** No. $\bar\sigma_T$ does not depend on $K$, so the time-0 implied-vol curve is flat across strike at every fixed expiry.

### (b)

Invert Black-Scholes to get $\sigma_{\text{imp}}(0.1),\sigma_{\text{imp}}(0.2),\sigma_{\text{imp}}(0.5)$ from the three observed call prices. Calibrate a piecewise-constant local volatility $\sigma(t)$ on $[0,0.1],[0.1,0.2],[0.2,0.5]$ using the L2.29 identity $\bar\sigma_T^2\,T=\int_0^T\sigma^2(t)\,dt$:
$$
\sigma_1^2 = \sigma_{\text{imp}}(0.1)^2,
\qquad
\sigma_2^2 = \frac{0.2\,\sigma_{\text{imp}}(0.2)^2 - 0.1\,\sigma_{\text{imp}}(0.1)^2}{0.1},
\qquad
\sigma_3^2 = \frac{0.5\,\sigma_{\text{imp}}(0.5)^2 - 0.2\,\sigma_{\text{imp}}(0.2)^2}{0.3}.
$$

In [12]:
def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_iv(S, K, T, r, price):
    return brentq(lambda s: bs_call(S, K, T, r, s) - price, 1e-6, 5.0)

S0_2, r_2, K_2 = 100.0, 0.05, 100.0
expiries = np.array([0.1, 0.2, 0.5])
prices   = np.array([5.25, 7.25, 9.5])
imps = np.array([bs_iv(S0_2, K_2, T, r_2, P) for T, P in zip(expiries, prices)])

sig1_sq = imps[0]**2
sig2_sq = (0.2*imps[1]**2 - 0.1*imps[0]**2) / 0.1
sig3_sq = (0.5*imps[2]**2 - 0.2*imps[1]**2) / 0.3

print("implied vols   :", imps)
print("sigma on [0.0,0.1]:", np.sqrt(sig1_sq))
print("sigma on [0.1,0.2]:", np.sqrt(sig2_sq))
print("sigma on [0.2,0.5]:", np.sqrt(sig3_sq))

implied vols   : [0.39732039 0.38017129 0.29509725]
sigma on [0.0,0.1]: 0.39732038579566276
sigma on [0.1,0.2]: 0.36221117165319183
sigma on [0.2,0.5]: 0.22087064039772794


### (c)

For the $T=0.4$ ATM call, the time-0 average squared volatility is
$$
\bar\sigma_{0.4}^2 = \frac{\sigma_1^2(0.1) + \sigma_2^2(0.1) + \sigma_3^2(0.2)}{0.4}
= \frac{0.2\,\sigma_{\text{imp}}(0.2)^2 + 0.2\,\sigma_3^2}{0.4},
$$
and the time-0 price is $C_{BS}(S_0,K,0.4,r,\bar\sigma_{0.4})$.

Viewed from time $0.1$, the call has $0.3$ years to expiry, and on $[0.1,0.4]$ the local vol pieces are $\sigma_2$ on $[0.1,0.2]$ and $\sigma_3$ on $[0.2,0.4]$. Since $\sigma$ depends only on $t$,
$$
\sigma_{\text{imp}}^{(t=0.1)}(K=100, T=0.4) = \sqrt{\frac{\sigma_2^2(0.1) + \sigma_3^2(0.2)}{0.3}},
$$
which is independent of $S_{0.1}$ and $K$.

In [13]:
sigbar_04 = np.sqrt((sig1_sq*0.1 + sig2_sq*0.1 + sig3_sq*0.2) / 0.4)
price_04 = bs_call(S0_2, K_2, 0.4, r_2, sigbar_04)

iv_t01 = np.sqrt((sig2_sq*0.1 + sig3_sq*0.2) / 0.3)

print("sigmabar(0.4)        :", sigbar_04)
print("time-0 price T=0.4   :", price_04)
print("time-0.1 IV of T=0.4 :", iv_t01)

sigmabar(0.4)        : 0.3108971298591373
time-0 price T=0.4   : 8.784201775931606
time-0.1 IV of T=0.4 : 0.27614284494912106
